# Music Annotation Project Notebook 62
## Page258 59R1-only / 59R2-retained probability-mass audit

**Runtime:** CPU. In Colab, select **Runtime → Run all**.

This notebook reads the durable scored parquet produced by Notebook 61 and answers one narrow question:

> For Page258 pixels that frozen 59R1 classified as coherent `SAFE_REMOVE`, but frozen 59R2 retained, where does the 59R2 calibrated `p_remove` probability mass sit?

It does **not** refit, recalibrate, rescore, or change either policy. It is a read-only diagnostic.

### Outputs
- exact counts and percentiles;
- fine probability bands, including the region immediately below `0.995`;
- separate analysis of pixels above `0.995` but rejected by 59R2 page-global coherence;
- distributions stratified by observed intensity, route evidence and component;
- spatial probability-band overlays on Page258;
- focused crops ranked by disagreement size and probability regime;
- a compact review ZIP that does not contain the large source parquet.

### Important limitation
The Notebook 61 parquet contains model evidence and decisions, not clean truth. “Light handwriting” is therefore approximated using observed greyscale intensity and spatial review. The notebook does not claim that every pale pixel is annotation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, shutil, sys, zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB61_ROOT = ROOT / '59r2_annotated_only_shadow_test'
NB61_RUN_OVERRIDE = None
PAGE = 'Page258'
EXPECTED_THRESHOLD = 0.995

OUTPUT_ROOT = ROOT / 'notebook62_page258_probability_audit'
RUN = OUTPUT_ROOT / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook62_page258_probability_audit')
TABLES = RUN / 'tables'
CHARTS = RUN / 'charts'
PAGE_OUT = RUN / 'page258'
CROPS = PAGE_OUT / 'component_crops'
AUDIT = RUN / 'audit'
UPLOAD = RUN / 'upload'
for p in [RUN, TABLES, CHARTS, PAGE_OUT, CROPS, AUDIT, UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)

# Intensity strata are descriptive, not truth labels.
INTENSITY_BANDS = [
    (-np.inf, 80, 'very_dark_<80'),
    (80, 140, 'dark_80_140'),
    (140, 180, 'midtone_140_180'),
    (180, 210, 'light_180_210'),
    (210, 235, 'very_light_210_235'),
    (235, np.inf, 'paper_like_>=235'),
]

# Fine bands deliberately isolate the region immediately below 0.995.
PROBABILITY_BANDS = [
    (0.995, 1.0000001, 'A_ge_0.995', '>= 0.995'),
    (0.990, 0.995, 'B_0.990_0.995', '0.990-0.995'),
    (0.980, 0.990, 'C_0.980_0.990', '0.980-0.990'),
    (0.950, 0.980, 'D_0.950_0.980', '0.950-0.980'),
    (0.900, 0.950, 'E_0.900_0.950', '0.900-0.950'),
    (0.500, 0.900, 'F_0.500_0.900', '0.500-0.900'),
    (0.000, 0.500, 'G_lt_0.500', '< 0.500'),
]

MAX_COMPONENT_CROPS = 24
CROP_MARGIN = 55
OVERLAY_ALPHA = 0.62

print('Output:', RUN)

## 2. Helpers and Notebook 61 discovery

In [ ]:
def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def atomic_json(path, value):
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def latest_complete(root, required):
    hits=[]
    root=Path(root)
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):
                hits.append(p)
    return max(hits, key=lambda p:p.stat().st_mtime) if hits else None

REQUIRED_NB61 = [
    'tables/59r2_scored_reviewed_cohort.parquet',
    'tables/59r1_vs_59r2_reviewed_pages.csv',
    'shadow_test_summary.json',
    'experiment_identity.json',
    f'pages/{PAGE}/observed.png',
    f'pages/{PAGE}/59r1_safe_remove_mask.png',
    f'pages/{PAGE}/59r2_safe_remove_mask.png',
]

NB61 = Path(NB61_RUN_OVERRIDE) if NB61_RUN_OVERRIDE else latest_complete(NB61_ROOT, REQUIRED_NB61)
require(NB61 is not None, 'Could not locate a complete Notebook 61 run. Set NB61_RUN_OVERRIDE.')

PARQUET_PATH = NB61 / 'tables' / '59r2_scored_reviewed_cohort.parquet'
OBSERVED_PATH = NB61 / 'pages' / PAGE / 'observed.png'
MASK59R1_PATH = NB61 / 'pages' / PAGE / '59r1_safe_remove_mask.png'
MASK59R2_PATH = NB61 / 'pages' / PAGE / '59r2_safe_remove_mask.png'
IDENTITY_PATH = NB61 / 'experiment_identity.json'
SUMMARY_PATH = NB61 / 'shadow_test_summary.json'

print('Notebook 61:', NB61)
print('Source parquet:', PARQUET_PATH)

## 3. Load only Page258 from the parquet

In [ ]:
# PyArrow predicate pushdown avoids loading all 12.65 million rows where supported.
page = pd.read_parquet(PARQUET_PATH, filters=[('page', '==', PAGE)])
if page.empty:
    # Fallback for parquet engines/files that cannot apply the filter.
    all_rows = pd.read_parquet(PARQUET_PATH)
    page = all_rows[all_rows.page.astype(str) == PAGE].copy()
    del all_rows

required = {
    'page','y','x','component_key','observed_value','paper_value',
    'p_remove','safe_remove','raw_remove_59r2','p_remove_59r2'
}
missing = required - set(page.columns)
require(not missing, f'Notebook 61 parquet lacks required columns: {sorted(missing)}')

page['page'] = page.page.astype(str)
for c in ['y','x','component_key']:
    page[c] = pd.to_numeric(page[c], errors='raise').astype(np.int32)
for c in ['observed_value','paper_value','p_remove','raw_remove_59r2','p_remove_59r2']:
    page[c] = pd.to_numeric(page[c], errors='raise').astype(np.float64)
page['safe_remove'] = page.safe_remove.astype(bool)

require(not page.duplicated(['y','x']).any(), 'Duplicate Page258 y/x rows found.')
require(np.isfinite(page[['p_remove','raw_remove_59r2','p_remove_59r2']].to_numpy()).all(), 'Non-finite probabilities found.')
require(((page.p_remove_59r2 >= 0) & (page.p_remove_59r2 <= 1)).all(), '59R2 probability outside [0,1].')

observed = cv2.imread(str(OBSERVED_PATH), cv2.IMREAD_GRAYSCALE)
mask1_png = cv2.imread(str(MASK59R1_PATH), cv2.IMREAD_GRAYSCALE)
mask2_png = cv2.imread(str(MASK59R2_PATH), cv2.IMREAD_GRAYSCALE)
require(observed is not None and mask1_png is not None and mask2_png is not None, 'Page258 images are unreadable.')
require(observed.shape == mask1_png.shape == mask2_png.shape, 'Page258 image/mask shape mismatch.')

h,w = observed.shape
y = page.y.to_numpy(); x = page.x.to_numpy()
require((y >= 0).all() and (x >= 0).all() and (y < h).all() and (x < w).all(), 'Page258 coordinate outside image.')

safe1_png = mask1_png[y,x] > 0
safe2 = mask2_png[y,x] > 0
require(np.array_equal(safe1_png, page.safe_remove.to_numpy()), 'Stored 59R1 mask does not match parquet safe_remove.')

page['safe_remove_59r1'] = safe1_png
page['safe_remove_59r2'] = safe2
page['population'] = np.select(
    [safe1_png & ~safe2, safe1_png & safe2, ~safe1_png & safe2],
    ['59R1_ONLY', 'BOTH', '59R2_ONLY'],
    default='NEITHER'
)

disagreement = page[page.population == '59R1_ONLY'].copy()
require(len(disagreement) > 0, 'No Page258 59R1-only disagreement pixels found.')

print('Page258 parquet rows:', f'{len(page):,}')
print(page.population.value_counts().to_string())

## 4. Reconstruct provisional threshold status and classify probability/intensity bands

In [ ]:
# The 59R2 final mask is read from Notebook 61. A pixel may be >=0.995 but still be absent
# from the final mask because page-global coherence rejected its connected region.
page['passes_59r2_pixel_threshold'] = page.p_remove_59r2 >= EXPECTED_THRESHOLD
page['above_threshold_but_coherence_rejected'] = page.passes_59r2_pixel_threshold & ~page.safe_remove_59r2

def probability_band(v):
    for lo,hi,band_id,label in PROBABILITY_BANDS:
        if lo <= v < hi:
            return band_id
    return 'UNASSIGNED'

def intensity_band(v):
    for lo,hi,label in INTENSITY_BANDS:
        if lo <= v < hi:
            return label
    return 'UNASSIGNED'

page['probability_band'] = [probability_band(v) for v in page.p_remove_59r2]
page['intensity_band'] = [intensity_band(v) for v in page.observed_value]
require(not (page.probability_band == 'UNASSIGNED').any(), 'Unassigned probability band.')
require(not (page.intensity_band == 'UNASSIGNED').any(), 'Unassigned intensity band.')

disagreement = page[page.population == '59R1_ONLY'].copy()

# Descriptive light-pixel subsets. These are not asserted to be handwriting truth.
disagreement['light_ge_180'] = disagreement.observed_value >= 180
disagreement['very_light_ge_210'] = disagreement.observed_value >= 210

page.to_parquet(TABLES/'page258_scored_with_populations.parquet', index=False)
disagreement.to_parquet(TABLES/'page258_59r1_only_59r2_retained.parquet', index=False)

## 5. Exact probability percentiles and band counts

In [ ]:
quantile_grid = [0, .001, .01, .05, .10, .25, .50, .75, .90, .95, .975, .99, .995, .999, 1.0]
populations = {
    '59R1_ONLY': page.population == '59R1_ONLY',
    'BOTH': page.population == 'BOTH',
    '59R2_ONLY': page.population == '59R2_ONLY',
    'NEITHER': page.population == 'NEITHER',
    '59R1_ONLY_LIGHT_GE_180': (page.population == '59R1_ONLY') & (page.observed_value >= 180),
    '59R1_ONLY_VERY_LIGHT_GE_210': (page.population == '59R1_ONLY') & (page.observed_value >= 210),
}

quantile_rows=[]
for name,mask in populations.items():
    vals=page.loc[mask,'p_remove_59r2']
    if len(vals)==0: continue
    for q,v in vals.quantile(quantile_grid).items():
        quantile_rows.append({'population':name,'rows':len(vals),'quantile':float(q),'p_remove_59r2':float(v)})
quantiles=pd.DataFrame(quantile_rows)
quantiles.to_csv(TABLES/'page258_probability_quantiles.csv',index=False)

band_counts=(disagreement.groupby(['probability_band'],as_index=False)
             .agg(pixels=('p_remove_59r2','size'),
                  mean_probability=('p_remove_59r2','mean'),
                  median_probability=('p_remove_59r2','median'),
                  mean_observed=('observed_value','mean'),
                  median_observed=('observed_value','median'),
                  components=('component_key','nunique')))
band_counts['fraction'] = band_counts.pixels / len(disagreement)
band_order=[b[2] for b in PROBABILITY_BANDS]
band_counts['order']=band_counts.probability_band.map({b:i for i,b in enumerate(band_order)})
band_counts=band_counts.sort_values('order').drop(columns='order')
band_counts.to_csv(TABLES/'page258_59r1_only_probability_bands.csv',index=False)

intensity_probability=(disagreement.groupby(['intensity_band','probability_band'],as_index=False)
                       .agg(pixels=('p_remove_59r2','size'),
                            components=('component_key','nunique'),
                            mean_probability=('p_remove_59r2','mean')))
intensity_probability.to_csv(TABLES/'page258_intensity_by_probability_band.csv',index=False)

threshold_summary={
    'page':PAGE,
    '59r1_only_pixels':int(len(disagreement)),
    'below_0_500':int((disagreement.p_remove_59r2 < .5).sum()),
    '0_500_to_0_900':int(((disagreement.p_remove_59r2 >= .5)&(disagreement.p_remove_59r2 < .9)).sum()),
    '0_900_to_0_950':int(((disagreement.p_remove_59r2 >= .9)&(disagreement.p_remove_59r2 < .95)).sum()),
    '0_950_to_0_980':int(((disagreement.p_remove_59r2 >= .95)&(disagreement.p_remove_59r2 < .98)).sum()),
    '0_980_to_0_990':int(((disagreement.p_remove_59r2 >= .98)&(disagreement.p_remove_59r2 < .99)).sum()),
    '0_990_to_0_995':int(((disagreement.p_remove_59r2 >= .99)&(disagreement.p_remove_59r2 < .995)).sum()),
    'at_or_above_0_995_but_coherence_rejected':int((disagreement.p_remove_59r2 >= .995).sum()),
    'light_ge_180_pixels':int((disagreement.observed_value >= 180).sum()),
    'very_light_ge_210_pixels':int((disagreement.observed_value >= 210).sum()),
    'median_probability':float(disagreement.p_remove_59r2.median()),
    'p90_probability':float(disagreement.p_remove_59r2.quantile(.90)),
    'p95_probability':float(disagreement.p_remove_59r2.quantile(.95)),
    'p99_probability':float(disagreement.p_remove_59r2.quantile(.99)),
    'maximum_probability':float(disagreement.p_remove_59r2.max()),
}
atomic_json(RUN/'page258_probability_mass_summary.json',threshold_summary)

print(json.dumps(threshold_summary,indent=2))
display(band_counts)
display(quantiles[quantiles.population.str.startswith('59R1_ONLY')])

## 6. Probability histograms and empirical CDF

In [ ]:
vals=disagreement.p_remove_59r2.to_numpy()
light=disagreement.loc[disagreement.observed_value>=180,'p_remove_59r2'].to_numpy()
very_light=disagreement.loc[disagreement.observed_value>=210,'p_remove_59r2'].to_numpy()

fig,axes=plt.subplots(2,2,figsize=(14,10))
axes[0,0].hist(vals,bins=np.linspace(0,1,101),color='#4472C4')
axes[0,0].axvline(.995,color='black',linestyle='--',label='0.995')
axes[0,0].set(title='All Page258 59R1-only pixels',xlabel='59R2 p_remove',ylabel='Pixels')
axes[0,0].legend()

fine_bins=np.unique(np.r_[np.linspace(.90,.98,33),np.linspace(.98,.995,31),np.linspace(.995,1,21)])
axes[0,1].hist(vals[vals>=.90],bins=fine_bins,color='#ED7D31')
axes[0,1].axvline(.995,color='black',linestyle='--')
axes[0,1].set(title='Upper tail: p_remove >= 0.90',xlabel='59R2 p_remove',ylabel='Pixels')

for arr,label,color in [(vals,'all disagreement','#4472C4'),(light,'observed >=180','#70AD47'),(very_light,'observed >=210','#A5A5A5')]:
    if len(arr):
        s=np.sort(arr); axes[1,0].plot(s,np.arange(1,len(s)+1)/len(s),label=f'{label} (n={len(s):,})',color=color)
axes[1,0].axvline(.995,color='black',linestyle='--')
axes[1,0].set(title='Empirical cumulative distribution',xlabel='59R2 p_remove',ylabel='Fraction <= probability')
axes[1,0].legend()

plot_counts=band_counts.copy()
axes[1,1].bar(plot_counts.probability_band,plot_counts.pixels,color='#5B9BD5')
axes[1,1].set(title='Fine probability-band counts',xlabel='Band',ylabel='Pixels')
axes[1,1].tick_params(axis='x',rotation=45)
for i,v in enumerate(plot_counts.pixels): axes[1,1].text(i,v,f'{int(v):,}',ha='center',va='bottom',fontsize=8)

plt.tight_layout(); plt.savefig(CHARTS/'page258_probability_mass_dashboard.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)

## 7. Spatial overlays: where each probability band occurs

In [ ]:
BGR = {
    'A_ge_0.995':(0,190,0),
    'B_0.990_0.995':(0,255,150),
    'C_0.980_0.990':(0,255,255),
    'D_0.950_0.980':(0,165,255),
    'E_0.900_0.950':(255,140,0),
    'F_0.500_0.900':(255,0,180),
    'G_lt_0.500':(80,80,80),
}
LABELS={b[2]:b[3] for b in PROBABILITY_BANDS}

def blend(gray, masks, alpha=OVERLAY_ALPHA):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); colour=base.copy(); active=np.zeros(gray.shape,bool)
    for mask,bgr in masks:
        colour[mask]=bgr; active |= mask
    out=base.copy(); out[active]=cv2.addWeighted(base,1-alpha,colour,alpha,0)[active]
    return out

def add_legend(image,title,items):
    h,w=image.shape[:2]; banner=78; canvas=np.full((h+banner,w,3),255,np.uint8); canvas[banner:]=image
    cv2.putText(canvas,title,(12,28),cv2.FONT_HERSHEY_SIMPLEX,.64,(30,30,30),1,cv2.LINE_AA)
    x0=12; y0=58
    for label,bgr in items:
        cv2.rectangle(canvas,(x0,y0-14),(x0+18,y0+4),bgr,-1)
        cv2.putText(canvas,label,(x0+24,y0+3),cv2.FONT_HERSHEY_SIMPLEX,.42,(30,30,30),1,cv2.LINE_AA)
        x0 += max(145, 8*len(label)+50)
    return canvas

masks=[]; items=[]
for band_id in [b[2] for b in PROBABILITY_BANDS]:
    m=np.zeros(observed.shape,bool)
    q=disagreement[disagreement.probability_band==band_id]
    m[q.y.to_numpy(int),q.x.to_numpy(int)]=True
    masks.append((m,BGR[band_id])); items.append((LABELS[band_id],BGR[band_id]))
overlay=add_legend(blend(observed,masks),'Page258: 59R1 removes, 59R2 retains — 59R2 p_remove bands',items)
cv2.imwrite(str(PAGE_OUT/'59r1_only_probability_band_overlay.png'),overlay)

# Light disagreement pixels only.
light_masks=[]
for band_id in [b[2] for b in PROBABILITY_BANDS]:
    m=np.zeros(observed.shape,bool)
    q=disagreement[(disagreement.observed_value>=180)&(disagreement.probability_band==band_id)]
    m[q.y.to_numpy(int),q.x.to_numpy(int)]=True
    light_masks.append((m,BGR[band_id]))
light_overlay=add_legend(blend(observed,light_masks),'Page258: observed >=180 within 59R1-only disagreement',items)
cv2.imwrite(str(PAGE_OUT/'light_pixels_probability_band_overlay.png'),light_overlay)

# Coherence-only rejection locator.
coh=np.zeros(observed.shape,bool)
q=disagreement[disagreement.p_remove_59r2>=EXPECTED_THRESHOLD]
coh[q.y.to_numpy(int),q.x.to_numpy(int)]=True
coh_overlay=add_legend(blend(observed,[(coh,(0,0,255))]),'Page258: p_remove >=0.995 but final 59R2 SAFE_REMOVE is false',[('coherence rejected',(0,0,255))])
cv2.imwrite(str(PAGE_OUT/'above_threshold_but_coherence_rejected.png'),coh_overlay)

display(overlay)

## 8. Component-level concentration and focused crops

In [ ]:
component_summary=(disagreement.groupby('component_key',as_index=False)
    .agg(pixels=('p_remove_59r2','size'),
         y_min=('y','min'),y_max=('y','max'),x_min=('x','min'),x_max=('x','max'),
         mean_probability=('p_remove_59r2','mean'),median_probability=('p_remove_59r2','median'),
         p90_probability=('p_remove_59r2',lambda s:float(s.quantile(.90))),
         p99_probability=('p_remove_59r2',lambda s:float(s.quantile(.99))),
         maximum_probability=('p_remove_59r2','max'),
         near_threshold_pixels=('p_remove_59r2',lambda s:int(((s>=.99)&(s<.995)).sum())),
         above_threshold_rejected=('p_remove_59r2',lambda s:int((s>=.995).sum())),
         light_pixels=('observed_value',lambda s:int((s>=180).sum())),
         very_light_pixels=('observed_value',lambda s:int((s>=210).sum())),
         mean_observed=('observed_value','mean')))
component_summary['near_threshold_fraction']=component_summary.near_threshold_pixels/component_summary.pixels
component_summary['light_fraction']=component_summary.light_pixels/component_summary.pixels
component_summary=component_summary.sort_values('pixels',ascending=False).reset_index(drop=True)
component_summary.to_csv(TABLES/'page258_disagreement_by_component.csv',index=False)

def crop_with_overlay(row, rank):
    x0=max(0,int(row.x_min)-CROP_MARGIN); y0=max(0,int(row.y_min)-CROP_MARGIN)
    x1=min(w,int(row.x_max)+CROP_MARGIN+1); y1=min(h,int(row.y_max)+CROP_MARGIN+1)
    comp=disagreement[disagreement.component_key==int(row.component_key)]
    local_masks=[]
    for band_id in [b[2] for b in PROBABILITY_BANDS]:
        m=np.zeros(observed.shape,bool); q=comp[comp.probability_band==band_id]
        m[q.y.to_numpy(int),q.x.to_numpy(int)]=True; local_masks.append((m,BGR[band_id]))
    left=cv2.cvtColor(observed[y0:y1,x0:x1],cv2.COLOR_GRAY2BGR)
    right=blend(observed,local_masks)[y0:y1,x0:x1]
    divider=np.full((left.shape[0],8,3),230,np.uint8)
    pair=np.hstack([left,divider,right])
    title=(f'rank={rank} component={int(row.component_key)} pixels={int(row.pixels):,} '
           f'median={row.median_probability:.4f} p99={row.p99_probability:.4f} '
           f'light={row.light_fraction:.1%}')
    canvas=np.full((pair.shape[0]+48,pair.shape[1],3),255,np.uint8); canvas[48:]=pair
    cv2.putText(canvas,title,(8,29),cv2.FONT_HERSHEY_SIMPLEX,.45,(25,25,25),1,cv2.LINE_AA)
    path=CROPS/f'{rank:02d}_component_{int(row.component_key)}.png'; cv2.imwrite(str(path),canvas)
    return path

crop_rows=[]
for rank,(_,row) in enumerate(component_summary.head(MAX_COMPONENT_CROPS).iterrows(),1):
    path=crop_with_overlay(row,rank); crop_rows.append({'rank':rank,'component_key':int(row.component_key),'crop_path':str(path),**row.to_dict()})
pd.DataFrame(crop_rows).to_csv(TABLES/'page258_component_crop_index.csv',index=False)

display(component_summary.head(20))

## 9. Decision-oriented interpretation table

In [ ]:
def classify_result(summary):
    n=summary['59r1_only_pixels']
    near=summary['0_990_to_0_995']/max(n,1)
    above=summary['at_or_above_0_995_but_coherence_rejected']/max(n,1)
    far=(summary['below_0_500']+summary['0_500_to_0_900'])/max(n,1)
    if near+above >= .50:
        return 'Most disagreement mass is at or immediately below the frozen threshold; calibration/threshold/coherence is the leading diagnostic issue.'
    if far >= .50:
        return 'Most disagreement mass is below 0.90; the model representation/ranking is the leading diagnostic issue, not a small threshold adjustment.'
    return 'The disagreement mass is mixed; inspect spatial bands and components rather than attributing the result to one threshold effect.'

interpretation=classify_result(threshold_summary)
summary_table=pd.DataFrame([
    {'question':'How many 59R1-only pixels?', 'answer':threshold_summary['59r1_only_pixels']},
    {'question':'Median 59R2 p_remove', 'answer':threshold_summary['median_probability']},
    {'question':'99th percentile 59R2 p_remove', 'answer':threshold_summary['p99_probability']},
    {'question':'Pixels in 0.990-0.995', 'answer':threshold_summary['0_990_to_0_995']},
    {'question':'Pixels >=0.995 but rejected by coherence', 'answer':threshold_summary['at_or_above_0_995_but_coherence_rejected']},
    {'question':'Pixels below 0.900', 'answer':threshold_summary['below_0_500']+threshold_summary['0_500_to_0_900']},
    {'question':'Observed >=180 pixels', 'answer':threshold_summary['light_ge_180_pixels']},
    {'question':'Diagnostic interpretation', 'answer':interpretation},
])
summary_table.to_csv(TABLES/'page258_decision_oriented_summary.csv',index=False)
display(summary_table)

## 10. Run identity and executive summary

In [ ]:
identity=json.loads(IDENTITY_PATH.read_text())
run_identity={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'Notebook62-Page258-probability-mass-audit-v1',
    'read_only_diagnostic':True,
    'retrained':False,'refitted':False,'recalibrated':False,'threshold_changed':False,
    'source_notebook61_run':str(NB61),
    'source_parquet':str(PARQUET_PATH),
    'source_parquet_sha256':sha256_file(PARQUET_PATH),
    'source_experiment_identity_sha256':sha256_file(IDENTITY_PATH),
    'policy_id':identity.get('policy_id'),
    'compatibility_id':identity.get('compatibility_id'),
    'page':PAGE,
    'threshold':EXPECTED_THRESHOLD,
    'rows':int(len(page)),
    '59r1_only_rows':int(len(disagreement)),
    'observed_intensity_is_not_truth':True,
    'promotion_authority':False,
    'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()},
}
atomic_json(RUN/'experiment_identity.json',run_identity)

lines=[
    'NOTEBOOK 62 — PAGE258 59R1-ONLY / 59R2-RETAINED PROBABILITY AUDIT','',
    f'Source Notebook 61 run: {NB61}',
    f'Policy ID: {identity.get("policy_id")}',
    f'Compatibility ID: {identity.get("compatibility_id")}',
    f'Page258 rows: {len(page):,}',
    f'59R1 removes / 59R2 retains: {len(disagreement):,}',
    f'Median 59R2 p_remove: {threshold_summary["median_probability"]:.6f}',
    f'90th percentile: {threshold_summary["p90_probability"]:.6f}',
    f'95th percentile: {threshold_summary["p95_probability"]:.6f}',
    f'99th percentile: {threshold_summary["p99_probability"]:.6f}',
    f'Maximum: {threshold_summary["maximum_probability"]:.6f}',
    f'Pixels in 0.990-0.995: {threshold_summary["0_990_to_0_995"]:,}',
    f'Pixels >=0.995 but rejected by coherence: {threshold_summary["at_or_above_0_995_but_coherence_rejected"]:,}',
    f'Pixels below 0.900: {threshold_summary["below_0_500"]+threshold_summary["0_500_to_0_900"]:,}',
    f'Observed >=180 disagreement pixels: {threshold_summary["light_ge_180_pixels"]:,}','',
    'DIAGNOSTIC INTERPRETATION',interpretation,'',
    'Observed intensity strata are descriptive only and are not annotation truth.',
    'This notebook changes no model, calibration, threshold, coherence rule, source image or frozen bundle.',
]
(RUN/'executive_summary.txt').write_text('
'.join(lines)+'
')
print('
'.join(lines))

## 11. Curated upload package

In [ ]:
curated=[
    RUN/'executive_summary.txt',
    RUN/'page258_probability_mass_summary.json',
    RUN/'experiment_identity.json',
    TABLES/'page258_probability_quantiles.csv',
    TABLES/'page258_59r1_only_probability_bands.csv',
    TABLES/'page258_intensity_by_probability_band.csv',
    TABLES/'page258_disagreement_by_component.csv',
    TABLES/'page258_component_crop_index.csv',
    TABLES/'page258_decision_oriented_summary.csv',
    CHARTS/'page258_probability_mass_dashboard.png',
    PAGE_OUT/'59r1_only_probability_band_overlay.png',
    PAGE_OUT/'light_pixels_probability_band_overlay.png',
    PAGE_OUT/'above_threshold_but_coherence_rejected.png',
]
curated.extend(sorted(CROPS.glob('*.png')))
missing=[str(p) for p in curated if not p.is_file()]
require(not missing, f'Missing curated files: {missing}')

manifest=[]
for src in curated:
    rel=src.relative_to(RUN); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
    require(sha256_file(src)==sha256_file(dst),f'Copy hash mismatch: {rel}')
    manifest.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).sort_values('relative_path').to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text(
    'Notebook 62 Page258 probability-mass diagnostic package.
'
    'Source parquet remains in the Notebook 61 run and is not copied into this ZIP.
'
    'Observed intensity is descriptive and is not annotation truth.
'
    'No model, calibrator, threshold, decision or source image was changed.
'
)
(UPLOAD/'upload_file_list.txt').write_text('
'.join(sorted(str(p.relative_to(UPLOAD)) for p in UPLOAD.rglob('*') if p.is_file()))+'
')

zip_path=RUN/'Notebook62_Page258_Probability_Mass_Audit_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file(): zf.write(p,p.relative_to(UPLOAD))
require(zip_path.is_file() and zip_path.stat().st_size>0,'ZIP creation failed.')
with zipfile.ZipFile(zip_path,'r') as zf:
    require(zf.testzip() is None,'ZIP integrity failed.')
print('Upload ZIP:',zip_path)
print('ZIP SHA-256:',sha256_file(zip_path))